In [1]:
# import sys
# !{sys.executable} -m pip install mat73

import numpy as np
import scipy
import mat73

In [1]:
from dataclasses import dataclass

## 1. Load Files

In [2]:
# Load MATLAB files
# For MATLAB v7.3 files, load with mat73 (https://github.com/skjerns/mat7.3)

time_binned_DiscreteSpikes = scipy.io.loadmat("./datafiles/C57_913_Qiu/time_binned_DiscreteSpikes.mat")
time_binned_SpikeInf = mat73.loadmat("./datafiles/C57_913_Qiu/time_binned_SpikeInf.mat")
target_positions = mat73.loadmat("./datafiles/C57_913_Qiu/target_positions.mat")
darktrial_raw = scipy.io.loadmat("./datafiles/C57_913_Qiu/darktrial_raw.mat")
del_trials = mat73.loadmat("./datafiles/C57_913_Qiu/del_trials.mat")

In [15]:
# Find the relevant keys to access data in the dictionaries
print(time_binned_SpikeInf.keys())
print(time_binned_DiscreteSpikes.keys())
print(target_positions.keys())
print(darktrial_raw.keys())
print(del_trials.keys())

dict_keys(['time_Caimg'])
dict_keys(['__header__', '__version__', '__globals__', 'time_Caimg'])
dict_keys(['position_tbins'])
dict_keys(['__header__', '__version__', '__globals__', 'darktrial'])
dict_keys(['del_trials'])


In [4]:
# Access data in dictionaries and assign data variable

spikeprob_tb100ms = time_binned_SpikeInf['time_Caimg']
discspikes_tb100ms = time_binned_DiscreteSpikes['time_Caimg']
positions_tb100ms = target_positions['position_tbins']
darktrials = darktrial_raw['darktrial']
deltrials_mat = del_trials['del_trials']

In [5]:
# Show data structure

# Spike Probability
print(spikeprob_tb100ms.shape)    # Neurons x Trials x T_Bin_100ms
# Discrete Spikes
print(discspikes_tb100ms.shape)   # Neurons x Trials x T_Bin_100ms
# Positions
print(positions_tb100ms.shape)    # Trials x T_Bin_100ms
# Trial Info
print(darktrials.shape)           # Trials (0=light, 1=dark)
print(deltrials_mat.shape)        # Trial_Num of deleted trials
print('no. of light trials: {}'.format(np.count_nonzero(darktrials==0)))
print('no. of dark trials: {}'.format(np.count_nonzero(darktrials)))

(127, 257, 223)
(127, 257, 223)
(257, 223)
(1, 257)
(13,)
no. of light trials: 157
no. of dark trials: 100


## 2. Preprocessing

- align axes across datasets
- convert position index from matlab format (count from 1) to python format (count from 0)
- remove deleted trials

In [6]:
# Align data matrices axes into: Trials x ____ x Neurons

spikeprob_tb100ms = spikeprob_tb100ms.swapaxes(0,1)
spikeprob_tb100ms = spikeprob_tb100ms.swapaxes(1,2)

discspikes_tb100ms = discspikes_tb100ms.swapaxes(0,1)
discspikes_tb100ms = discspikes_tb100ms.swapaxes(1,2)

darktrials = darktrials.swapaxes(0,1)

print(spikeprob_tb100ms.shape)
print(discspikes_tb100ms.shape)
print(darktrials.shape)

(257, 223, 127)
(257, 223, 127)
(257, 1)


In [7]:
# Convert deltrials indices to integers that start from 0

print(deltrials_mat)
deltrials = np.array([int(x - 1) for x in deltrials_mat])
print(deltrials)

[  1.   2.   3.   4.  58.  77.  87.  89. 116. 173. 177. 253. 257.]
[  0   1   2   3  57  76  86  88 115 172 176 252 256]


In [8]:
# Change position bins from matlab format to python

print(np.nanmin(positions_tb100ms), np.nanmax(positions_tb100ms))

# matlab counts from 1, pythong counts from 0
positions_tb100ms -= 1

print(np.nanmin(positions_tb100ms), np.nanmax(positions_tb100ms))

1.0 50.0
0.0 49.0


In [9]:
# Remove deleted trials

spikeprob_tb100ms = np.delete(spikeprob_tb100ms, deltrials, axis=0)
discspikes_tb100ms = np.delete(discspikes_tb100ms, deltrials, axis=0)
posbins_tb100ms = np.delete(positions_tb100ms, deltrials, axis=0)
darktrials = np.delete(darktrials, deltrials, axis=0)

In [10]:
# Remove non-neuron data, n=0 and n=1 (background and neuropil)

# In upstream data processing, ROI 0 and ROI 1 are not neurons. They are the baseline
# from the background and the neuropil.

spikeprob_tb100ms = np.delete(spikeprob_tb100ms, [0,1], axis=2)
discspikes_tb100ms = np.delete(discspikes_tb100ms, [0,1], axis=2)

In [11]:
# Match NaN values between spikeprob and discspikes

# Due to artefacts from upstream data processing pipeline, there are arbitrary
# data in discspikes when it should have been NaN. To get rid of the artefact,
# we create a NaN mask from spikeprob and apply it to discspikes.

print("nans before matching:", np.sum(np.isnan(discspikes_tb100ms)))

# True = NaN
nanmask = np.isnan(spikeprob_tb100ms)
discspikes_tb100ms[nanmask] = np.nan
    
print("nans after matching:", np.sum(np.isnan(discspikes_tb100ms)))

nans before matching: 3565750
nans after matching: 4244625


In [ ]:
# Remove position_mtx artefacts

# Due to artefacts from upstream data processing pipeline, there are occasions
# in position_mtx where there are zeroes when it should have been NaNs. To get rid
# of the artefact, we replace these zeroes with NaNs.

# List of indices with position 0
ls_of_pos0 = list(zip(*np.where(positions_tb100ms == 0)))

for i in ls_of_pos0:      
    trial, pos0_tbin = i    
    # find the last non-NaN index of the trial
    last_nonnan_tbin = np.where(~np.isnan(positions_tb100ms[trial]))[0][-1]
    
    # if the position 0 time bin is also the last non-NaN time bin of the trial,
    # which is likely an artefact, it will be replaced with NaN
    if pos0_tbin == last_nonnan_tbin:
        positions_tb100ms[i] = np.nan

In [24]:
def show_data():
    # Spike Probability
    print("Spike Probability:")
    print(spikeprob_tb100ms.shape)
    print("Trial x 100ms Time Bin x Neuron")
    print()

    # Discrete Spikes
    print("Discrete Spikes:")
    print(discspikes_tb100ms.shape)
    print("Trial x 100ms Time Bin x Neuron")
    print()

    # Position Matrix
    print("Position Matrices:")
    print(positions_tb100ms.shape)
    print("Trial x 100ms Time Bin")
    print()

    # Trial Info
    print("Dark Trials:")
    print(darktrials.shape)
    print("Trial, (0=light, 1=dark)")
    print()
    print("Deleted Trials:")
    print(deltrials.shape)
    print("Trial, (Trial_Num of deleted trials)")
    print()

    print('no. of light trials: {}'.format(np.count_nonzero(darktrials==0)))
    print('no. of dark trials: {}'.format(np.count_nonzero(darktrials)))

In [25]:
show_data()

Spike Probability:
(244, 223, 125)
Trial x 100ms Time Bin x Neuron

Discrete Spikes:
(244, 223, 125)
Trial x 100ms Time Bin x Neuron

Position Matrices:
(257, 223)
Trial x 100ms Time Bin

Dark Trials:
(244, 1)
Trial, (0=light, 1=dark)

Deleted Trials:
(13,)
Trial, (Trial_Num of deleted trials)

no. of light trials: 150
no. of dark trials: 94
